In [ ]:
!pip install timm Augmentor -q

In [ ]:
import os, shutil, json, torch, timm
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm
import Augmentor

In [ ]:
class ParallelBlock(nn.Module):
    def __init__(self):
        super().__init__()

        self.b1 = nn.Conv2d(3,64,3,padding=1)
        self.b2 = nn.Conv2d(3,64,5,padding=2)
        self.b3 = nn.Conv2d(3,64,7,padding=3)

    def forward(self,x):
        return torch.cat([
            torch.relu(self.b1(x)),
            torch.relu(self.b2(x)),
            torch.relu(self.b3(x))
        ],dim=1)

In [ ]:
class DeepCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.parallel = ParallelBlock()

        self.conv1 = nn.Sequential(
            nn.Conv2d(192,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU()
        )

        self.pool1 = nn.MaxPool2d(2)

        self.conv2 = nn.Sequential(
            nn.Conv2d(256,512,3,padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU()
        )

        self.pool2 = nn.MaxPool2d(2)

        self.conv3 = nn.Sequential(
            nn.Conv2d(512,512,3,padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU()
        )

    def forward(self,x):
        x = self.parallel(x)
        x = self.pool1(self.conv1(x))
        x = self.pool2(self.conv2(x))
        x = self.conv3(x)
        return x

In [ ]:
class Classifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.encoder = timm.create_model(
            "vit_base_patch16_224",
            pretrained=False,
            num_classes=0
        )

        self.encoder.load_state_dict(torch.load("/kaggle/input/datasets/avanishkumar4309/best-encoder/best_encoder (3).pth"))

        self.cnn = DeepCNN()

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512*4*4,512),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(512,128),
            nn.ReLU(),

            nn.Linear(128,num_classes)
        )

    def forward(self,x):
        x = self.encoder(x)
        x = x.view(x.size(0),3,16,16)
        x = self.cnn(x)
        return self.fc(x)

In [ ]:
data_dir = "/kaggle/input/datasets/avanishkumar4309/finaledataset"

full_ds = datasets.ImageFolder(data_dir)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.3,0.3,0.3,0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [ ]:
train_size = int(0.8*len(full_ds))
val_size = len(full_ds) - train_size

train_ds, val_ds = random_split(full_ds,[train_size,val_size])

train_ds.dataset.transform = train_tf
val_ds.dataset.transform = val_tf

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)

print("Classes:", full_ds.classes)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import json

# -------- LOAD MODEL --------
model = Classifier(len(full_ds.classes)).to(device)

checkpoint = torch.load("/kaggle/input/datasets/avanishkumar4309/checkpoints/checkpoint.pth")

model.load_state_dict(checkpoint['model'])
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.AdamW(model.parameters(), lr=1e-4)
optimizer.load_state_dict(checkpoint['optimizer'])

start_epoch = checkpoint['epoch'] + 1

print(f"🔁 Resuming from epoch {start_epoch}")

# -------- BEST ACC TRACK --------
best_acc = 0

# -------- TRAIN LOOP --------
for epoch in range(start_epoch, 10):

    model.train()
    correct, total = 0, 0

    loop = tqdm(train_loader)

    for imgs, labels in loop:
        imgs, labels = imgs.to(device), labels.to(device)

        out = model(imgs)
        loss = criterion(out, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        _, pred = torch.max(out, 1)
        total += labels.size(0)
        correct += (pred == labels).sum().item()

        loop.set_postfix(acc=100 * correct / total)

    epoch_acc = 100 * correct / total
    print(f"Epoch {epoch} Acc: {epoch_acc:.2f}%")

    # -------- 🔥 SAVE CHECKPOINT (HAR EPOCH) --------
    torch.save({
        'epoch': epoch,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict()
    }, "checkpoint.pth")

    print("💾 Checkpoint saved")

    # -------- 🔥 SAVE BEST MODEL --------
    if epoch_acc > best_acc:
        best_acc = epoch_acc
        torch.save(model.state_dict(), "best_model.pth")
        print("🏆 Best model saved")

    # -------- 🔥 AUTO SAVE FINAL EVERY EPOCH --------
    torch.save(model.state_dict(), "final_model.pth")

    with open("classes.json", "w") as f:
        json.dump(full_ds.classes, f)

    print("✅ Auto-saved model + classes")

print("🎯 TRAINING COMPLETE")

In [ ]:
model.load_state_dict(torch.load("final_model.pth"))
model.eval()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

y_true = []
y_pred = []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        out = model(imgs)

        _, preds = torch.max(out, 1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=full_ds.classes,
            yticklabels=full_ds.classes)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
print(classification_report(y_true, y_pred, target_names=full_ds.classes))

In [ ]:
accuracy = np.mean(np.array(y_true) == np.array(y_pred))
print("Final Accuracy:", accuracy*100)

In [ ]:
train_acc_list = []

In [ ]:
train_acc_list.append(100*correct/total)

In [ ]:
plt.plot(train_acc_list)
plt.title("Training Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.show()

In [ ]:
import numpy as np

# 1. Calculate accuracy for each class
# cm.diagonal() gives correct predictions, cm.sum(axis=1) gives total per class
class_accuracies = cm.diagonal() / cm.sum(axis=1)

# 2. Print the results clearly
print("--- Class-wise Accuracy ---")
for i, class_name in enumerate(full_ds.classes):
    accuracy = class_accuracies[i] * 100
    print(f"{class_name:30} : {accuracy:.2f}%")
